What Stage 5 does and why it's structured this wayStage 5 converts the continuous density field from SIMP into a manufacturable STL file. This is harder than it sounds — the density field lives on an unstructured tetrahedral mesh, but marching cubes (the standard isosurface extraction algorithm) expects a structured voxel grid. The bridge between these two representations is the core problem this stage solves.The data flow for this stage is:outputs/meshes/<name>_stage04.json      ← handoff from Stage 4
outputs/meshes/<name>_density.xdmf     ← density field on tet mesh
        ↓
voxelization: tet mesh → regular grid   ← scattered interpolation
marching cubes (scikit-image)           ← isosurface at rho=0.5
mesh repair (trimesh)                   ← fill holes, fix normals, remove islands
        ↓
outputs/stl/<name>_optimized.stl        ← final export
outputs/reports/<name>_stl_*.png        ← before/after renders
outputs/meshes/<name>_stage05.json      ← pipeline completion recordThere is no src/ module for this stage — the logic is self-contained enough that splitting it would add indirection without benefit. Everything lives in the notebook cells, backed by scikit-image and trimesh which are already in the container.

Cell 0 — Parameters (tag: parameters)

In [8]:
# Cell 0 — tagged: parameters
import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")

PART_NAME_OVERRIDE = None   # set by pipeline_full.ipynb in sweep mode

STAGE04_HANDOFF  = None     # auto-detect if None
DENSITY_THRESHOLD = 0.65     # isosurface level — rho > this is "solid"
                            # raise to 0.6 for heavier/safer result
                            # lower to 0.4 for lighter/more aggressive
SMOOTH_ITERATIONS = 50      # Laplacian smoothing passes on final mesh
                            # 0 = raw marching cubes, 10+ = very rounded
REMOVE_ISLANDS    = True    # drop disconnected components smaller than 1%
                            # of total mesh volume

RENDER_PLOTS = False

Cell 1 — Load Stage 4 handoff

In [9]:
# Cell 1 — Load Stage 4 handoff
from pathlib import Path
import json
import numpy as np

# Use opt_domain mesh for export
xdmf_path    = Path("outputs/meshes/opt_domain.xdmf")
density_path = Path("outputs/meshes/base_part_density.xdmf")
part_name = PART_NAME_OVERRIDE if PART_NAME_OVERRIDE else "base_part"

assert xdmf_path.exists(), f"Not found: {xdmf_path}"
assert density_path.exists(), f"Not found: {density_path}"
print(f"Mesh:    {xdmf_path}")
print(f"Density: {density_path}")

Mesh:    outputs/meshes/opt_domain.xdmf
Density: outputs/meshes/base_part_density.xdmf


Cell 2 — Load density field and mesh geometry

In [10]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Load mesh and density field
# ════════════════════════════════════════════════════════════
 
import h5py
import numpy as np
from pathlib import Path
from mpi4py import MPI
from dolfinx.io import XDMFFile
 
comm = MPI.COMM_WORLD
 
# Load mesh from the optimization domain
with XDMFFile(comm, str(xdmf_path), "r") as xdmf:
    domain = xdmf.read_mesh(name="Grid")
 
# Auto-detect coordinate units
coord_max = domain.geometry.x.max()
if coord_max > 1.0:
    domain.geometry.x[:] /= 1000.0  # mm → m
 
# Load density values from HDF5
h5_path = str(density_path).replace(".xdmf", ".h5")
with h5py.File(h5_path, "r") as f:
    rho_values = f["Function/density/0"][:].ravel()
 
# Node coordinates in mm for STL export
pts_mm = domain.geometry.x * 1000.0
 
# Element connectivity
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, 0)
conn   = domain.topology.connectivity(tdim, 0)
n_elem = domain.topology.index_map(tdim).size_local
tets   = np.array([conn.links(i) for i in range(n_elem)])
 
print(f"Mesh nodes:      {len(pts_mm):,}")
print(f"Elements:        {n_elem:,}")
print(f"Density range:   [{rho_values.min():.4f}, {rho_values.max():.4f}]")
print(f"Above threshold ({DENSITY_THRESHOLD}): "
      f"{(rho_values > DENSITY_THRESHOLD).sum():,} "
      f"({100*(rho_values > DENSITY_THRESHOLD).mean():.1f}%)")
print(f"Near void (<0.1):  {(rho_values < 0.1).mean()*100:.1f}%")
print(f"Near solid (>0.9): {(rho_values > 0.9).mean()*100:.1f}%")
 

Mesh nodes:      9,261
Elements:        41,489
Density range:   [0.0010, 1.0000]
Above threshold (0.65): 26,288 (63.4%)
Near void (<0.1):  36.1%
Near solid (>0.9): 63.3%


Cell 3 — Voxelize density field

In [11]:
print(f"Using xdmf_path: {xdmf_path}")
print(f"Using density_path: {density_path}")
print(f"DENSITY_THRESHOLD: {DENSITY_THRESHOLD}")

solid_mask = rho_values > DENSITY_THRESHOLD
solid_tets  = tets[solid_mask]
print(f"Solid elements: {solid_mask.sum():,} / {n_elem:,} "
      f"({100*solid_mask.mean():.1f}%)")

# Each tet has 4 faces. For face index fi, the opposite vertex index is opp_vi.
# The outward normal must point AWAY from the opposite vertex (which is interior).
# This determines winding at extraction time, eliminating post-hoc fix_normals stalls.
TET_FACE_COMBOS = [[0,1,2],[0,1,3],[0,2,3],[1,2,3]]
TET_OPP_VERTEX  = [    3,      2,      1,      0  ]   # opposite vertex for each face

# Step 1: find boundary face keys (count==1 across all solid tets)
all_faces_raw  = solid_tets[:, TET_FACE_COMBOS].reshape(-1, 3)
sorted_faces   = np.sort(all_faces_raw, axis=1)
_, inverse, counts = np.unique(sorted_faces, axis=0,
                                return_inverse=True, return_counts=True)
boundary_mask  = counts[inverse] == 1   # True for each face row that is a boundary face

# Step 2: for each boundary face, determine outward winding using the opposite vertex
# Build a map from sorted-face key → (tet_local_verts, opposite_vertex_coord)
boundary_faces_final = []
seen_keys = set()

for fi_flat, is_bnd in enumerate(boundary_mask):
    if not is_bnd:
        continue
    tet_idx  = fi_flat // 4
    face_idx = fi_flat % 4
    tet      = solid_tets[tet_idx]
    fc       = TET_FACE_COMBOS[face_idx]
    opp_i    = TET_OPP_VERTEX[face_idx]

    key = tuple(sorted_faces[fi_flat])
    if key in seen_keys:
        continue
    seen_keys.add(key)

    v0, v1, v2 = pts_mm[tet[fc[0]]], pts_mm[tet[fc[1]]], pts_mm[tet[fc[2]]]
    opp_v      = pts_mm[tet[opp_i]]
    normal     = np.cross(v1 - v0, v2 - v0)
    face_ctr   = (v0 + v1 + v2) / 3.0

    # If normal points toward the opposite (interior) vertex, flip winding
    if np.dot(normal, face_ctr - opp_v) > 0:
        # Normal already points away from interior vertex — outward
        boundary_faces_final.append([tet[fc[0]], tet[fc[1]], tet[fc[2]]])
    else:
        # Normal points inward — flip by swapping last two vertices
        boundary_faces_final.append([tet[fc[0]], tet[fc[2]], tet[fc[1]]])

boundary_faces_final = np.array(boundary_faces_final)
print(f"Boundary faces: {len(boundary_faces_final):,} (outward-oriented at extraction)")

Using xdmf_path: outputs/meshes/opt_domain.xdmf
Using density_path: outputs/meshes/base_part_density.xdmf
DENSITY_THRESHOLD: 0.65
Solid elements: 26,288 / 41,489 (63.4%)
Boundary faces: 7,192


Cell 4 — Build trimesh, repair, smooth and export STL

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Build trimesh, repair, smooth and export STL
# ════════════════════════════════════════════════════════════

import trimesh
import numpy as np
from pathlib import Path
from collections import defaultdict

# ── Construct mesh ────────────────────────────────────────────────────────
raw_mesh = trimesh.Trimesh(
    vertices=pts_mm,
    faces=boundary_faces_final,
    process=False,
)

# Diagnostic: check open edges immediately after construction
nm_initial = trimesh.grouping.group_rows(raw_mesh.edges_sorted, require_count=1)
print(f"After construction: {len(raw_mesh.vertices):,} verts, "
      f"{len(raw_mesh.faces):,} faces, {len(nm_initial)} open edges")

# ── Step 1: Merge duplicate vertices ─────────────────────────────────────
raw_mesh.merge_vertices()
nm = trimesh.grouping.group_rows(raw_mesh.edges_sorted, require_count=1)
print(f"After merge_vertices: {len(raw_mesh.vertices):,} verts, "
      f"{len(nm)} open edges")

# ── Step 2: Fix normals ───────────────────────────────────────────────────
trimesh.repair.fix_normals(raw_mesh)
trimesh.repair.fix_winding(raw_mesh)
nm = trimesh.grouping.group_rows(raw_mesh.edges_sorted, require_count=1)
print(f"After fix_normals: {len(nm)} open edges")

# Assign mesh for further processing
mesh = raw_mesh
print(f"Watertight before smoothing: {mesh.is_watertight}")

# ── Step 3: Smooth ────────────────────────────────────────────────────────
if SMOOTH_ITERATIONS > 0:
    trimesh.smoothing.filter_taubin(
        mesh,
        lamb=0.5,
        nu=0.53,
        iterations=SMOOTH_ITERATIONS,
    )
    nm = trimesh.grouping.group_rows(mesh.edges_sorted, require_count=1)
    print(f"After {SMOOTH_ITERATIONS} Taubin iterations: {len(nm)} open edges")

print(f"Watertight after smoothing: {mesh.is_watertight}")

# ── Step 4: Clean up ─────────────────────────────────────────────────────
mesh.update_faces(mesh.nondegenerate_faces())
mesh.update_faces(mesh.unique_faces())
mesh.remove_unreferenced_vertices()

# ── Step 5: Remove disconnected islands ──────────────────────────────────
if REMOVE_ISLANDS:
    components = mesh.split(only_watertight=False)
    if len(components) > 1:
        vols = [abs(c.volume) for c in components]
        total_vol = sum(vols)
        # Keep any component with >1% of total volume
        large = [c for c, v in zip(components, vols) if v > 0.01 * total_vol]
        n_dropped = len(components) - len(large)
        if large and n_dropped > 0:
            mesh = trimesh.util.concatenate(large)
            print(f"  Removed {n_dropped} island(s) "
                  f"(<1% vol each, {sum(v for c,v in zip(components,vols) if v <= 0.01*total_vol):.1f} mm³ total)")
            print(f"  Kept {len(large)} component(s)")
        elif n_dropped == 0:
            print(f"  No islands to remove ({len(components)} component(s), all ≥1% vol)")
    else:
        print("  Mesh is a single component — no islands")

# ── Final report ──────────────────────────────────────────────────────────
nm_final = trimesh.grouping.group_rows(mesh.edges_sorted, require_count=1)
print(f"\nFinal:      {len(mesh.vertices):,} verts, {len(mesh.faces):,} faces")
print(f"Open edges: {len(nm_final)}")
print(f"Watertight: {mesh.is_watertight}")
print(f"Volume:     {abs(mesh.volume):.1f} mm³")
if mesh.bounds is not None:
    print(f"Bounds X:   [{mesh.bounds[0][0]:.1f}, {mesh.bounds[1][0]:.1f}] mm")
    print(f"Bounds Y:   [{mesh.bounds[0][1]:.1f}, {mesh.bounds[1][1]:.1f}] mm")
    print(f"Bounds Z:   [{mesh.bounds[0][2]:.1f}, {mesh.bounds[1][2]:.1f}] mm")

if not mesh.is_watertight:
    print(f"⚠ Not watertight — {len(nm_final)} open edges remain")

# ── Export STL ────────────────────────────────────────────────────────────
output_stl_dir = Path("outputs/stl")
output_stl_dir.mkdir(parents=True, exist_ok=True)
stl_path = output_stl_dir / f"{part_name}_optimized.stl"
mesh.export(str(stl_path))

stl_size_kb = stl_path.stat().st_size / 1024
print(f"\nSTL exported: {stl_path}")
print(f"File size:    {stl_size_kb:.1f} KB")
print(f"Triangles:    {len(mesh.faces):,}")

After construction: 9,261 verts, 7,192 faces, 0 open edges
After merge_vertices: 3,576 verts, 0 open edges
After fix_normals: 0 open edges
Watertight before smoothing: False
After 50 Taubin iterations: 0 open edges
Watertight after smoothing: False

Final:      3,576 verts, 7,192 faces
Open edges: 0
Watertight: False
Volume:     66787.3 mm³
Bounds X:   [-0.4, 100.4] mm
Bounds Y:   [-0.5, 60.5] mm
Bounds Z:   [-2.1, 22.1] mm
⚠ Not watertight — 0 open edges remain

STL exported: outputs/stl/base_part_optimized.stl
File size:    351.3 KB
Triangles:    7,192


Cell 7 — Before/after renders

In [13]:
# Cell 7 — Side-by-side render: original STL vs optimized STL
if not RENDER_PLOTS:
    print("Skipping render — set RENDER_PLOTS=True after fixing display")
else:
    import pyvista as pv
    from pathlib import Path
    pv.OFF_SCREEN = True

    report_dir = Path("outputs/reports")

    original_stl = Path(f"outputs/meshes/{part_name}.stl")

    pl = pv.Plotter(shape=(1, 2), window_size=(1600, 600))
    if original_stl.exists():
        orig_mesh = pv.read(str(original_stl))
        pl.subplot(0, 0)
        pl.add_mesh(orig_mesh, color="lightsteelblue", show_edges=False, opacity=0.9)
        pl.add_text(
            f"Original\n{orig_mesh.n_cells:,} faces",
            font_size=10,
        )
        pl.view_isometric()
    else:
        pl.subplot(0, 0)
        pl.add_text("Original STL not found", font_size=10)

    opt_mesh = pv.read(str(stl_path))
    pl.subplot(0, 1)
    pl.add_mesh(opt_mesh, color="coral", show_edges=False, opacity=0.9)
    pl.add_text(
        f"Optimized (vf={0.4:.2f})\n{opt_mesh.n_cells:,} faces",
        font_size=10,
    )
    pl.view_isometric()

    compare_png = report_dir / f"{part_name}_before_after.png"
    pl.screenshot(str(compare_png))
    pl.close()
    print(f"Before/after render: {compare_png}")

    pl2 = pv.Plotter()
    pl2.add_mesh(opt_mesh, style="wireframe", color="steelblue", line_width=0.5)
    pl2.view_isometric()
    wire_png = report_dir / f"{part_name}_stl_wireframe.png"
    pl2.screenshot(str(wire_png))
    pl2.close()
    print(f"Wireframe render:    {wire_png}")

Skipping render — set RENDER_PLOTS=True after fixing display


Cell 8 — Pipeline completion record

In [ ]:
# Cell 8 — Write final stage05 record and pipeline summary
#
# This is the terminal handoff — no further stages consume it.
# It's a complete audit record of the full pipeline run:
# parameters, timings, and output paths for every stage.
import json
from pathlib import Path
from datetime import datetime

# Gather all stage handoffs for the summary
stage_handoffs = {}
for stage_json in sorted(Path("outputs/meshes").glob(f"{part_name}_stage0*.json")):
    stage_data = json.loads(stage_json.read_text())
    stage_handoffs[stage_data["stage"]] = stage_data

_s04 = stage_handoffs.get("04_simp", {})

record = {
    "stage":          "05_stl_export",
    "part_name":      part_name,
    "completed_at":   datetime.now(datetime.UTC).isoformat(),
    "stl_path":       str(stl_path),
    "stl_size_kb":    round(stl_size_kb, 2),
    "mesh": {
        "vertices":   len(mesh.vertices),
        "triangles":  len(mesh.faces),
        "watertight": mesh.is_watertight,
        "volume_mm3": round(abs(mesh.volume), 3),
        "area_mm2":   round(mesh.area, 3),
    },
    "export_config": {
        "density_threshold": DENSITY_THRESHOLD,
        "smooth_iterations": SMOOTH_ITERATIONS,
        "remove_islands":    REMOVE_ISLANDS,
    },
    "pipeline_summary": {
        s: {
            "duration_s": v.get("duration_s"),
        }
        for s, v in stage_handoffs.items()
    },
    "optimization": {
        "volume_fraction":  _s04.get("config", {}).get("volume_fraction"),
        "n_iterations":     _s04.get("n_iterations"),
        "converged":        _s04.get("converged"),
        "final_compliance": _s04.get("final_compliance"),
    },
}
record_path = Path("outputs/meshes") / f"{part_name}_stage05.json"
record_path.write_text(json.dumps(record, indent=2))
print("═" * 60)
print("  PIPELINE COMPLETE")
print("═" * 60)
print(f"  Part:          {part_name}")
print(f"  STL:           {stl_path}")
print(f"  Watertight:    {mesh.is_watertight}")
print(f"  Volume:        {abs(mesh.volume):.2f} mm³")
print(f"  Triangles:     {len(mesh.faces):,}")
_vf = _s04.get("config", {}).get("volume_fraction")
print(f"  Material saved: {(1 - _vf) * 100:.1f}%" if _vf is not None else "  Material saved: —")
print("─" * 60)
print("  Stage durations:")
for stage, data in stage_handoffs.items():
    dur = data.get("duration_s", "—")
    print(f"    {stage:<20} {dur}s")
print("═" * 60)
print(f"\n  Full record: {record_path}")

════════════════════════════════════════════════════════════
  PIPELINE COMPLETE
════════════════════════════════════════════════════════════
  Part:          base_part
  STL:           outputs/stl/base_part_optimized.stl
  Watertight:    False
  Volume:        66787.29 mm³
  Triangles:     7,192
  Material saved: 55.0%
────────────────────────────────────────────────────────────
  Stage durations:
    01_geometry          0.251s
    02_meshing           —s
    03_fea               —s
    04_simp              54.864s
    05_stl_export        —s
════════════════════════════════════════════════════════════

  Full record: outputs/meshes/base_part_stage05.json
